In [4]:
import os
import pandas as pd

# Navigate UP to project root
project_root = '/home/surya/honeypot-project'
csv_path = os.path.join(project_root, 'data', 'events.csv')

print("Full CSV path:", csv_path)
print("File exists?", os.path.exists(csv_path))

# Load it
df = pd.read_csv(csv_path)
print(f"✅ Loaded {len(df)} events")
print(df[['source_ip', 'command', 'event_type']].head())


Full CSV path: /home/surya/honeypot-project/data/events.csv
File exists? True
✅ Loaded 13 events
   source_ip            command event_type
0    8.8.8.8            sudo rm        ssh
1    8.8.8.8            sudo rm        ssh
2    8.8.8.8            sudo rm        ssh
3    8.8.8.8             whoami        ssh
4  127.0.0.1  web_login_attempt        web


In [5]:
from collections import Counter
import numpy as np

# Features for your honeypot data
df['cmd_len'] = df['command'].fillna('').str.len()
df['sudo_flag'] = df['command'].fillna('').str.contains('sudo', case=False, na=False).astype(int)
df['web_event'] = (df['event_type'] == 'web').astype(int)

# IP frequency (how often each IP attacks)
ip_counts = Counter(df['source_ip'])
df['ip_freq'] = df['source_ip'].map(ip_counts)

# Label: 1=malicious (sudo cmds, ssh attacks, non-Surya logins)
df['label'] = np.where(
    (df['command'].fillna('').str.contains('sudo|rm|whoami', case=False, na=False)) |
    (df['event_type'] == 'ssh') | 
    (df['username'] != 'Surya'), 1, 0
)

print("✅ Features added!")
print(df[['source_ip', 'ip_freq', 'cmd_len', 'sudo_flag', 'web_event', 'label']].head(10))
print("\nMalicious count:", df['label'].value_counts())


✅ Features added!
   source_ip  ip_freq  cmd_len  sudo_flag  web_event  label
0    8.8.8.8        4        7          1          0      1
1    8.8.8.8        4        7          1          0      1
2    8.8.8.8        4        7          1          0      1
3    8.8.8.8        4        6          0          0      1
4  127.0.0.1        7       17          0          1      1
5   evil.com        1       13          1          0      1
6  127.0.0.1        7       17          0          1      0
7  127.0.0.1        7       17          0          1      0
8  127.0.0.1        7       13          0          1      0
9  127.0.0.1        7       13          0          1      0

Malicious count: label
1    8
0    5
Name: count, dtype: int64


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib

# Features + target
features = ['ip_freq', 'cmd_len', 'sudo_flag', 'web_event']
X = df[features].fillna(0)
y = df['label']

# Split + train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)
score = model.score(X_test, y_test)

print(f"✅ Trained on {len(df)} events")
print(f"Test accuracy: {score:.2f}")
print("Feature importance:", dict(zip(features, model.feature_importances_)))


✅ Trained on 13 events
Test accuracy: 0.75
Feature importance: {'ip_freq': 0.3620502156095064, 'cmd_len': 0.22470607372060286, 'sudo_flag': 0.07388020143122183, 'web_event': 0.3393635092386688}


In [7]:
import joblib
import os

# Ensure models/ dir exists
os.makedirs('../models', exist_ok=True)

# Save trained model for FastAPI /predict
joblib.dump(model, '../models/honeypot_enhanced_model.pkl')
print("✅ SAVED honeypot_enhanced_model.pkl")
print("Model ready for production!")
print(f"Features used: {features}")


✅ SAVED honeypot_enhanced_model.pkl
Model ready for production!
Features used: ['ip_freq', 'cmd_len', 'sudo_flag', 'web_event']


In [8]:
# Test load works
test_model = joblib.load('../models/honeypot_enhanced_model.pkl')
print("✅ Model loads correctly!")
print("Ready for /predict endpoint & Railway!")


✅ Model loads correctly!
Ready for /predict endpoint & Railway!
